In [ ]:
# 1/5 - Clone TokenVector prototype/cuda-backend (source .tkv + do compiler sinh .cu)
import subprocess, os
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
import glob
print('examples:', [os.path.basename(x) for x in glob.glob(d+'/Release/cuda_backend/examples/*')])

In [ ]:
# 2/5 - Kiem tra file .cu DO COMPILER SINH (cuda_feature.tkv), khong dung Python generator
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/kernel_demo.cuda.cu'
src=open(cu).read()
print('generated .cu exists =', os.path.exists(cu))
print('has __global__ =', '__global__' in src)
print('has launcher     =', 'tkv_launch_vec_add' in src)
print('has sync         =', 'cudaDeviceSynchronize' in src)
assert 'cudaDeviceSynchronize' in src, 'launcher chua co sync -> Kernel khong goi duoc du lieu ve host'
print(src[:1600])

In [ ]:
# 3/5 - Build .so bang nvcc (file .cu do compiler sinh)
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/kernel_demo.cuda.cu'
so=ex+'/tkv_cuda_kernels.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
print('nvcc =', nvcc)
cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu '
     '-Xcompiler -std=c++20 -Xcompiler -fPIC '
     +cu+' -o '+so+' -lcudart')
p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
print('build rc=', p.returncode)
if p.returncode!=0: print('STDERR:', p.stderr[:4000])
print('so exists=', os.path.exists(so))

In [ ]:
# 4/5 - Load .so, goi tkv_launch_vec_add (do .tkv dinh nghia) voi DEVICE pointers
import ctypes, torch, time
so='/content/TokenVectorCuda/Release/cuda_backend/examples/tkv_cuda_kernels.so'
lib=ctypes.CDLL(so)
F32=ctypes.c_float; I32=ctypes.c_int; I64=ctypes.c_int64
lib.tkv_launch_vec_add.argtypes=[I64,I64,I64,I32,I32,I32]   # device ptr as int64
lib.tkv_launch_vec_add.restype=I32
n=1<<20
a=torch.rand(n,device='cuda',dtype=torch.float32)
b=torch.rand(n,device='cuda',dtype=torch.float32)
c=torch.zeros(n,device='cuda',dtype=torch.float32)
pa=a.data_ptr(); pb=b.data_ptr(); pc=c.data_ptr()   # CUDA device addresses
grid=(n+255)//256; block=256
t0=time.perf_counter(); rc=lib.tkv_launch_vec_add(pa,pb,pc,n,grid,block); dt=(time.perf_counter()-t0)*1000
err=(c-(a+b)).abs().max().item()
print('vec_add rc=',rc,' time_ms=',round(dt,3))
print('vec_add max_abs_err=',err,' CORRECT=',err<1e-5)

In [ ]:
# 5/5 - Summary
print('CUDA BACKEND (TokenVector .tkv -> cuda_feature.tkv -> .cu -> nvcc -> launcher) TEST COMPLETE')
print('If CORRECT=True: kernel dinh nghia bang TokenVector chay dung tren GPU.')
print('Next: viet fused AdamW kernel bang .tkv de thay doan AdamW C++ trong torch_fused.cpp.')